In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from math import pi
import os
from pathlib import Path

path_cwd=Path.cwd()
path_input = str(path_cwd) + '/Output_files/'


column_names = ['Date', 'Rain', 'PET', 'SV', 'Total SF (cm3/h)', 'S_avg', 'PSY']
sf_df = pd.read_csv(path_input + 'DF27-2021-ALL.csv', header=None, skiprows=1, names=column_names)
sf_df.index = pd.to_datetime(sf_df['Date'])
sf_df = sf_df.drop(columns=['Date'])

dates=sf_df.index

In [2]:
#Lists from field data
# Environmental 
Precipitation_2021= sf_df.filter(items=['Rain'])
PET_2021= sf_df.filter(items=['PET']) #mm/h

# Sap Flow
SF_2021=sf_df.filter(items=['Total SF (cm3/h)'])
# Soil Moisture
SM_2021=sf_df.filter(items=['S_avg'])

In [3]:
# Fracture size and leaf area

#grow the surface fracture 
surface_area=0.7#0.7 #m2 Needs to be at least tree area ~7000 cm2- this makes sense for a a 32 cm fracture with times 2.18 m (which is what we have roughly for 27)
surface_depth=0.1#0.1 #m 10-30 cm Surface fracture 
leaf_area= 10.83 #m2 leaf area of a tree, to calculate it we used the sapwood area=32.82178925 cm2 and a literature value for Al:Asw of 0.45 m2/cm2
tree_diameter= 0.32 #m diameter of the tree
branch_length= 1.3 # m assuming longest branch is 1.3 
projected_area= pi*(branch_length+(tree_diameter/2))**2 # m2 projecte area (shadow circle over ground)
LAI=round(leaf_area/projected_area) #leaf area index  
pt_corr= LAI/5 # Deng et al 2017 assumes PT/PET = LAI/LAImax LAImax was calculated 

In [4]:
#Adjusting the data to the fracture size

#environmental 
precipitation=((Precipitation_2021['Rain'].values)*surface_area).tolist() # mm/h is over 1m2 so if we want over a smaller area we need to multiply by the area
pet=((PET_2021['PET'].values)*surface_area).tolist() #mm/h calculated every 30 min and assumed to be over 1m2 -> mm~L if 0.7m2 -> mm*0.7~L

#sap flow
sf_orig=(((SF_2021['Total SF (cm3/h)'].values)/(leaf_area*1e4))*10).tolist() #cm3/h/cm2 of leaf area --> cm/h *10=mm/h sapflux normalized by leaf area 

#soil moisture
sm_vwc_orig = (((SM_2021['S_avg'].values))).tolist() # m3/m3
sm_orig = (((SM_2021['S_avg'].values)*(surface_area*surface_depth))*1000).tolist() # mm 

wilting_point= 0.05*(surface_area*surface_depth)*1000 #0.05-0.1 mm https://www.researchgate.net/figure/Field-capacity-FC-a-permanent-wilting-point-PWP-b-and-available-water-capacity_fig3_332182471
saturation_= 0.4*(surface_area*surface_depth)*1000 # 0.1-0.2 mm but data shows a bit higher values



In [5]:
# General simulation elements stocks and flows

# Number of time steps
Nt = len(Precipitation_2021) # total half hours 

# Reservoirs-Stocks [L]
deep_fracture = [None]*Nt 
surface_fracture = [None]*Nt 
quickflow= [None]*Nt 
#we can potentially add flow out to transpiration and evaporation


# Flows- Record all flow channels
q1 = [None]*Nt # precipitation -> quickflow (overflow)
q2 = [None]*Nt # precipitation -> surface (input)
q3 = [None]*Nt # surface -> deep (infiltration)
q4 = [None]*Nt # surface -> atmosphere/tree (evaporation+transpiration surface)
q5 = [None]*Nt # deep -> quickflow (seepage)
q6 = [None]*Nt # deep -> atmosphere/tree (transpiration deep)

# Flows to save 
infiltration=[None]*Nt
seepage=[None]*Nt
# rwu=[None]*Nt

In [6]:
#Initial and boundary conditions

# Initial conditions
quickflow [0] = 0# mm
deep_fracture[0] = 1 # mm any value between 0 and df_max (in this case df_max=13.99 so value<=13 is fine)
surface_fracture[0] = sm_orig[0] # mm 15 because that's what the data shows for the first row of season 

# Box bounds 
sf_max= saturation_ #maximum capacity for surface fracture mm ~28 mm 
sf_min= wilting_point #minimum capacity for surface fracture mm ~3.5 mm 
df_max= saturation_/2# maximum capacity for deep fracture mm /half of surface fracture
df_min= 0.01 #minimum capacity for deep fracture mm

In [7]:
#PARAMETERS FOR FLOWS

#inflow surface
x_qf=0.2 #fraction of precipitation that goes to quickflow if surface fracture is not full # [0.7-0.9] for type of crack 27 is growing in 

#outflow infiltration surface-deep
f_sd_d = 0.00025#0.00025 # % or h-1-- initial (max) guess for surface-deep percolation coefficient #0.00125 h-1 = 0.03/24 ; 0.03 day-1 is Husic et al. (2019) median value for percolation coefficient bounded by quickflow (0.11) and zero flow
f_adjust = 0.00005# amount to decrease f_sd by if condition is not met- keep it an order of magnitude smaller than f_sd_d

#outflow seepage deep-quickflow
f_dqf=0.0000046#0.0046 #% if h-1: 0.0046 #0.0046 h-1 0.11/24 if percolation as quickflow (empties fast)- if not it should be smaller/ regular 0.00060 h-1 / poor drainage 0.0000046 h-1

#outflow evaporation surface-atmosphere
f_ev= 0.4#0.5 evaporated fraction #assume it constant throughout the simulation- technically it should change with temperature and humidity
f_rwu_surface=0.2 # transpired fraction 

#outflow deep-transpiration 
f_rwu_deep=0.9 # transpired fraction

In [8]:
#adjusting precipitation (more or less)
precipitation=[i for i in ((Precipitation_2021['Rain'].values)*surface_area).tolist()] #mm/h


In [9]:

for n in range(1,Nt):
    surface_fracture[n] = surface_fracture[n-1]
    deep_fracture[n] = deep_fracture[n-1]
    quickflow[n] = quickflow[n-1]
    
    ########################################################################################################################################################
    # INFLOW to Surface 
    # Law: fraction lost to quickflow and fraction into soil 
    # Running only this law will keep all the boxes constant except for the surface fracture which will grow with precipitation
    if precipitation[n] > 0:
        if surface_fracture[n] < sf_max:
            if surface_fracture[n]+precipitation[n] > sf_max: #esto evita ponding 
                X=1
            else: 
                X=x_qf
        else: 
            X=1
        q1[n] = precipitation[n]*X # quickflow loss 
        q2[n] = precipitation[n]*(1-X) #water input to surface fracture 
        surface_fracture[n] += q2[n]
        quickflow[n] += q1[n]
    ########################################################################################################################################################

    # OUTFLOW from Surface

    ########################################################################################################################################################
    # INFLOW to deep
    #Law: fraction infiltrated and adjusted based on deep fracture capacity
    # Running this and the previous law will keep all the boxes constant except for deep fracture that will grow and surface fracture will decrease with infiltration
    # with only these 2 boxes the system will saturate both boxes we need seepage and evaporation to keep the system in balance
    if (surface_fracture[n]>sf_min): 
        surface_fracture_copy = surface_fracture[n]  
        deep_fracture_copy = deep_fracture[n]  
        while True:
            q3_d = f_sd_d * (surface_fracture_copy - sf_min) #if it's percolation: percolation coefficient * dt * (surface_fracture_copy - sf_min)
            DF_d = deep_fracture_copy + q3_d
            if DF_d < df_max:
                f_sd = f_sd_d
                break  
            f_sd_d -= f_adjust  # If the condition is not met, decrease f_sd by amount
    else:
        f_sd=0 #no infiltration
    
    #Law: infiltration surface-deep
    q3[n] = f_sd *(surface_fracture[n]-sf_min) # height grad by multiplied by adjusted percolation coefficient [f_sd]=h-1
    deep_fracture[n] += q3[n]
    surface_fracture[n] -= q3[n]
    infiltration[n]=q3[n] #mm/h
   


    ########################################################################################################################################################
    # INFLOW to atmosphere 
    # Law: evaporation fractions surface-atmosphere
    # Running this additional to the previous laws will decrease the surface fracture through evaporation but the deep is still saturated cause there's no seepage
    if (surface_fracture[n]>sf_min): 
        if (precipitation[n]>0):
            q4[n] = 0
        else:
            q4[n] = (surface_fracture[n]/sf_max)*pet[n]*f_ev + (surface_fracture[n]/sf_max)*pet[n]*f_rwu_surface
        surface_fracture[n] -= q4[n]


    ########################################################################################################################################################

    # OUTFLOW from Deep

    ########################################################################################################################################################
    # Outflow out of sysem
    #Law: Seepage deep-quickflow
    if (deep_fracture[n]>df_min): 
        q5[n] = f_dqf*(deep_fracture[n]-df_min) 
    else:
        q5[n]=0

    quickflow[n] += q5[n]
    deep_fracture[n] -= q5[n]
    seepage[n]=q5[n]  #mm/h

    # INFLOW to atmosphere 
    # Law: tree uptake deep-atmosphere
    if (deep_fracture[n]>df_min) & (precipitation[n]==0): 
        q6[n] = (deep_fracture[n]/df_max)*pet[n]*f_rwu_deep 
        #q6[n]=0

        deep_fracture[n] -= q6[n]


    ########################################################################################################################################################
    # COPY TO NEXT TIMESTEP
    if n < (Nt-1):
        surface_fracture[n+1] = surface_fracture[n]
        deep_fracture[n+1] = deep_fracture[n]
        quickflow [n+1] = quickflow[n]


#Calculate Nash‐Sutcliffe efﬁciency (NSE)
#NSE=1-(sum((Qobs-Qsim)^2)/sum((Qobs-Qobs_mean)^2))

nse = 1-sum((np.array(sm_vwc_orig)-np.array([i/(surface_area*surface_depth*1000) for i in surface_fracture]))**2)/sum((np.array(sm_vwc_orig)-np.mean(sm_vwc_orig))**2)

#Calculate Kling-Gupta efﬁciency (KGE)
#KGE=1-(sqrt((rho-1)^2+(alpha-1)^2+(beta-1)^2))
#rho=correlation coefficient between simulated and observed values
#alpha= ratio of the mean of simulated to observed values
#beta= ratio of the standard deviation of simulated to observed values

kge = 1-np.sqrt((np.corrcoef(sm_vwc_orig,[i/(surface_area*surface_depth*1000) for i in surface_fracture])[0,1]-1)**2+(np.mean([i/(surface_area*surface_depth*1000) for i in surface_fracture])/np.mean(sm_vwc_orig)-1)**2+(np.std([i/(surface_area*surface_depth*1000) for i in surface_fracture])/np.std(sm_vwc_orig)-1)**2)



fig = go.Figure()
#stocks 
# fig.add_trace(go.Scatter(x=dates, y=[i for i in surface_fracture],
#                     mode='lines',
#                     name='Volume Surface (mm)-simulated'))

# ##from data directly 
# fig.add_trace(go.Scatter(x=dates, y=sm_orig, 
#                     name='Soil Moisture(mm)-data'))

# fig.add_trace(go.Scatter(x=dates, y=deep_fracture,
#                     mode='lines',
#                     name='Volume Deep (mm)'))


# fig.add_trace(go.Scatter(x=dates, y=[i for i in precipitation],
#                     name='Precipitation (mm h-1)-data'))

#change volume to volumetric water content
fig.add_trace(go.Scatter(x=dates, y=[i/(surface_area*surface_depth*1000) for i in surface_fracture],
                    mode='lines',
                    name='Surface Simulated (m3/m3)'))

fig.add_trace(go.Scatter(x=dates, y=sm_vwc_orig,
                    name='Surface Data (m3/m3)'))

fig.add_trace(go.Scatter(x=dates, y=[i/(surface_area*surface_depth*1000) for i in deep_fracture],
                    mode='lines',
                    name='Deep Simulated (m3/m3)'))

fig.add_hline(y=0.1, line_dash="dot",
              annotation_text="PWP", 
              annotation_position="bottom right")

fig.update_layout(title='Regular precipitation; f_E='+str(f_ev) + '; f_T='+str(f_rwu_surface)+'; NSE='+str(round(nse,2))+'; KGE='+str(round(kge,2)),xaxis_title='Time', yaxis_title='Volumetric Water Content (m3/m3)')

#calculate percent of surface simulated that is below wilting point
percent_below_wp = len([i for i in surface_fracture if i/(surface_area*surface_depth*1000)<0.1])/len(surface_fracture)
print(percent_below_wp)
fig.show()

0.4866811633596086
